<a href="https://www.kaggle.com/code/avikdas567/predict-customer-churn?scriptVersionId=301268881" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
import random
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# SEED EVERYTHING

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# LOAD DATA

train_orig = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test_orig = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv")

TARGET = "Churn"
ID = "id"

print("Train shape:", train_orig.shape)
print("Test shape:", test_orig.shape)

# COPY DATA

train = train_orig.copy()
test = test_orig.copy()

# Convert target to numeric
y = train[TARGET].map({"No": 0, "Yes": 1})
train.drop(columns=[TARGET], inplace=True)

# LABEL ENCODING FOR LGB + XGB

full = pd.concat([train, test], axis=0).reset_index(drop=True)
cat_cols = full.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    full[col] = le.fit_transform(full[col].astype(str))

train_enc = full.iloc[:len(y)].reset_index(drop=True)
test_enc = full.iloc[len(y):].reset_index(drop=True)

# STRATIFIED K-FOLD

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

oof_lgb = np.zeros(len(train_enc))
oof_xgb = np.zeros(len(train_enc))
oof_cat = np.zeros(len(train_orig))

test_preds_lgb = np.zeros(len(test_enc))
test_preds_xgb = np.zeros(len(test_enc))
test_preds_cat = np.zeros(len(test_orig))

# LIGHTGBM

print("\nTraining LightGBM...")

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.03,
    "num_leaves": 64,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 1,
    "lambda_l2": 1,
    "device": "gpu",
    "verbosity": -1,
    "seed": SEED
}

for fold, (train_idx, val_idx) in enumerate(skf.split(train_enc, y)):
    print(f"LGB Fold {fold+1}")
    
    X_train, X_val = train_enc.iloc[train_idx], train_enc.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params, n_estimators=3000)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(200)]
    )
    
    oof_lgb[val_idx] = model.predict_proba(X_val)[:,1]
    test_preds_lgb += model.predict_proba(test_enc)[:,1] / n_splits

print("LGB AUC:", roc_auc_score(y, oof_lgb))

# XGBOOST

print("\nTraining XGBoost...")

for fold, (train_idx, val_idx) in enumerate(skf.split(train_enc, y)):
    print(f"XGB Fold {fold+1}")
    
    X_train, X_val = train_enc.iloc[train_idx], train_enc.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        device="cuda",
        n_estimators=3000,
        random_state=SEED
    )
    
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    oof_xgb[val_idx] = model.predict_proba(X_val)[:,1]
    test_preds_xgb += model.predict_proba(test_enc)[:,1] / n_splits

print("XGB AUC:", roc_auc_score(y, oof_xgb))

# CATBOOST

print("\nTraining CatBoost...")

train_cb = train_orig.copy()
test_cb = test_orig.copy()

y_cb = train_cb[TARGET].map({"No": 0, "Yes": 1})
train_cb.drop(columns=[TARGET], inplace=True)

cat_cols_cb = train_cb.select_dtypes(include=["object"]).columns.tolist()

cat_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "learning_rate": 0.03,
    "depth": 6,
    "iterations": 3000,
    "random_seed": SEED,
    "task_type": "GPU",
    "verbose": False
}

for fold, (train_idx, val_idx) in enumerate(skf.split(train_cb, y_cb)):
    print(f"CAT Fold {fold+1}")
    
    X_train = train_cb.iloc[train_idx]
    X_val = train_cb.iloc[val_idx]
    y_train = y_cb.iloc[train_idx]
    y_val = y_cb.iloc[val_idx]
    
    model = CatBoostClassifier(**cat_params)
    
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_cols_cb,
        early_stopping_rounds=200,
        verbose=False
    )
    
    oof_cat[val_idx] = model.predict_proba(X_val)[:,1]
    test_preds_cat += model.predict_proba(test_cb)[:,1] / n_splits

print("CAT AUC:", roc_auc_score(y_cb, oof_cat))

# ENSEMBLE

print("\nCreating Ensemble...")

oof_ensemble = 0.4 * oof_lgb + 0.3 * oof_xgb + 0.3 * oof_cat
test_ensemble = 0.4 * test_preds_lgb + 0.3 * test_preds_xgb + 0.3 * test_preds_cat

print("ENSEMBLE AUC:", roc_auc_score(y, oof_ensemble))

# SAVE SUBMISSION

submission = pd.DataFrame({
    ID: sample_sub[ID],
    TARGET: test_ensemble
})

submission.to_csv("submission.csv", index=False)
print("\nSubmission saved as submission.csv")

Train shape: (594194, 21)
Test shape: (254655, 20)

Training LightGBM...
LGB Fold 1


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[770]	valid_0's auc: 0.915713
LGB Fold 2
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[650]	valid_0's auc: 0.91677
LGB Fold 3
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[888]	valid_0's auc: 0.916241
LGB Fold 4
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[780]	valid_0's auc: 0.917222
LGB Fold 5
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[826]	valid_0's auc: 0.914658
LGB AUC: 0.9161108442511752

Training XGBoost...
XGB Fold 1
XGB Fold 2
XGB Fold 3
XGB Fold 4
XGB Fold 5
XGB AUC: 0.9155262916954159

Training CatBoost...
CAT Fold 1


Default metric period is 5 because AUC is/are not implemented for GPU


CAT Fold 2


...

CAT Fold 4


Default metric period is 5 because AUC is/are not implemented for GPU


CAT Fold 5


Default metric period is 5 because AUC is/are not implemented for GPU


CAT AUC: 0.9158438764122757

Creating Ensemble...
ENSEMBLE AUC: 0.9163970780271148

Submission saved as submission.csv
